# GNN Training Demo

This notebook demonstrates training a Graph Neural Network for photonic band gap prediction.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import torch
import matplotlib.pyplot as plt
from torch_geometric.loader import DataLoader

from src.data.photonic_crystal_generator import PhotonicCrystalGenerator
from src.data.graph_converter import CrystalToGraphConverter
from src.models.gnn_bandgap import GNNBandGapPredictor
from src.dft.band_structure_calculator import BandStructureCalculator
from src.utils.visualization import plot_prediction_vs_actual

## Generate Training Data

In [ ]:
# Set random seed
torch.manual_seed(42)
np.random.seed(42)

# Generate crystals
generator = PhotonicCrystalGenerator(random_seed=42)
graph_converter = CrystalToGraphConverter()
band_calc = BandStructureCalculator()

# Generate small dataset for demo
num_samples = 50
print(f"Generating {num_samples} training samples...")

crystals = generator.generate_dataset(num_samples, grid_size=32)

In [ ]:
# Calculate band gaps and convert to graphs
graph_dataset = []
band_gaps = []

for i, crystal in enumerate(crystals):
    if i % 10 == 0:
        print(f"Processing crystal {i+1}/{num_samples}")
    
    try:
        band_gap_info = band_calc.calculate_band_gap(crystal)
        band_gap = band_gap_info['band_gap']
        
        graph_data = graph_converter.crystal_to_graph(crystal, band_gap)
        graph_dataset.append(graph_data)
        band_gaps.append(band_gap)
    except Exception as e:
        print(f"Error: {e}")
        continue

print(f"\nSuccessfully processed {len(graph_dataset)} samples")
print(f"Band gap range: [{min(band_gaps):.4f}, {max(band_gaps):.4f}]")

## Inspect Graph Data

In [ ]:
# Inspect first graph
sample_graph = graph_dataset[0]

print("Sample Graph:")
print(f"  Number of nodes: {sample_graph.num_nodes}")
print(f"  Number of edges: {sample_graph.edge_index.shape[1]}")
print(f"  Node features shape: {sample_graph.x.shape}")
print(f"  Edge features shape: {sample_graph.edge_attr.shape}")
print(f"  Target band gap: {sample_graph.y.item():.6f}")
print(f"  Lattice type: {sample_graph.lattice_type}")

## Split Dataset

In [ ]:
# Split into train/val
val_size = int(len(graph_dataset) * 0.2)
train_size = len(graph_dataset) - val_size

indices = np.random.permutation(len(graph_dataset))
train_dataset = [graph_dataset[i] for i in indices[:train_size]]
val_dataset = [graph_dataset[i] for i in indices[train_size:]]

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

## Create Data Loaders

In [ ]:
batch_size = 8

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Number of training batches: {len(train_loader)}")
print(f"Number of validation batches: {len(val_loader)}")

## Initialize Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = GNNBandGapPredictor(
    node_feature_dim=6,
    edge_feature_dim=1,
    hidden_dim=64,
    num_conv_layers=3,
    num_fc_layers=2,
    dropout=0.2,
    pooling='attention'
).to(device)

print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")

## Training Setup

In [ ]:
import torch.nn as nn
from torch.optim import Adam

criterion = nn.MSELoss()
optimizer = Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

# Training history
history = {
    'train_loss': [],
    'val_loss': []
}

## Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        outputs = model(batch)
        predictions = outputs['band_gap']
        
        loss = criterion(predictions, batch.y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch.num_graphs
    
    return total_loss / len(loader.dataset)

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            outputs = model(batch)
            predictions = outputs['band_gap']
            
            loss = criterion(predictions, batch.y)
            total_loss += loss.item() * batch.num_graphs
            
            all_preds.extend(predictions.cpu().numpy())
            all_targets.extend(batch.y.cpu().numpy())
    
    return total_loss / len(loader.dataset), np.array(all_preds), np.array(all_targets)

In [ ]:
# Train for a few epochs
num_epochs = 20

print("Starting training...\n")
for epoch in range(num_epochs):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_preds, val_targets = validate(model, val_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}")
        print(f"  Train Loss: {train_loss:.6f}")
        print(f"  Val Loss:   {val_loss:.6f}\n")

print("Training complete!")

## Visualize Training History

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(history['train_loss'], 'b-', label='Training Loss', linewidth=2)
plt.plot(history['val_loss'], 'r-', label='Validation Loss', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('MSE Loss', fontsize=12)
plt.title('Training History', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.show()

## Evaluate Predictions

In [ ]:
fig = plot_prediction_vs_actual(
    val_targets,
    val_preds,
    title="Validation Set: Predicted vs Actual Band Gap"
)
plt.show()

## Test on New Crystal

In [ ]:
# Generate a new test crystal
test_crystal = generator.generate_2d_square_lattice(
    n_background=1.0,
    n_rod=3.0,
    lattice_constant=0.6,
    rod_radius=0.35,
    grid_size=32
)

# Calculate true band gap
true_bg_info = band_calc.calculate_band_gap(test_crystal)
true_band_gap = true_bg_info['band_gap']

# Predict with GNN
test_graph = graph_converter.crystal_to_graph(test_crystal)
test_graph = test_graph.to(device)

model.eval()
with torch.no_grad():
    outputs = model(test_graph)
    predicted_band_gap = outputs['band_gap'].item()

print("Single Crystal Prediction:")
print(f"  True band gap:      {true_band_gap:.6f}")
print(f"  Predicted band gap: {predicted_band_gap:.6f}")
print(f"  Absolute error:     {abs(true_band_gap - predicted_band_gap):.6f}")
print(f"  Relative error:     {abs(true_band_gap - predicted_band_gap) / true_band_gap * 100:.2f}%")

## Summary

This notebook demonstrated:
- Converting photonic crystals to graph representations
- Training a GNN model for band gap prediction
- Evaluating model performance
- Making predictions on new crystals

For production training with larger datasets, use the `train.py` script:
```bash
python train.py --num_samples 1000 --epochs 100
```